In [4]:
import json
import pandas as pd

# Load CARD JSON
with open("card.json", "r") as f:
    card_data = json.load(f)

prefixes = [
    "IMI",
    "KPC",
    "TEM",
    "SHV",
    "CTX-M",
    "VEB",
    "SME",
    "GES",
    "OXA",
    "NDM",
    "ADC without carbapenemase activity",
    "PDC",
    "EC",
    "ACT",
    "DHA",
    "MOX",
    "CMY",
    "FOX"
]

carbapenemase_families = [
    "IMI",
    "KPC",
    "SME",
    "GES",
    "NDM",
    "PDC"
]

data = []

for entry_id, entry in card_data.items():

    # Skip non-dictionary entries (metadata etc.)
    if not isinstance(entry, dict):
        continue

    model_name = entry.get("model_name", "")

    # Skip entries without model_name
    if not model_name:
        continue

    # Check gene prefix
    if not any(model_name.startswith(prefix) for prefix in prefixes):
        continue

    model_sequences = entry.get("model_sequences", {})
    sequence_block = model_sequences.get("sequence", {})

    for seq_id, seq_entry in sequence_block.items():

        protein_seq = (
            seq_entry.get("protein_sequence", {})
            .get("sequence", "")
        )

        if protein_seq:
            data.append({
                "Gene": model_name,
                "Sequence": protein_seq
            })

# Create dataframe
df = pd.DataFrame(data)

# Extract family
df["Family"] = df["Gene"].str.extract(r"^(CTX-M|IMI|KPC|TEM|SHV|VEB|SME|ADC|GES|OXA|NDM|PDC|EC|amp-C|ACT|DHA|MOX|CMY|FOX)")[0]

# Label carbapenemases
df["Carbapenemase"] = df["Family"].isin(carbapenemase_families).astype(int)

# Keep columns
df = df[["Family", "Gene", "Sequence", "Carbapenemase"]]

# Save
df.to_csv("card_sequences.csv", index=False)
df.head()


,Family,Gene,Sequence,Carbapenemase
0,SHV,SHV-52,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,0
1,CTX-M,CTX-M-130,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,0
2,NDM,NDM-6,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRF...,1
3,ACT,ACT-35,MMKKSLCCALLLGISCSALATPVSEKQLAEVVANTVTPLMKAQSVP...,0
4,TEM,TEM-126,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,0
